# 01 — Extraction [ETL Build (Raw → Cleaned Dataset)]
*Use this notebook to load the raw dataset, inspect its structure, and confirm that the source is suitable for the project.*

---

This notebook is the **single source of truth** for extraction + cleaning output generation.  
It executes the same production pipeline used by `scripts/etl_pipeline.py` so output remains identical.

| | |
|---|---|
| **Input** | `data/raw/*.csv` (9 Olist files) |
| **Output** | `data/processed/cleaned_master.csv` |
| **Goal** | Reproducible build — same schema and rows every run |


## 1. Setup

In [2]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)

RAW_DIR  = PROJECT_ROOT / 'data' / 'raw'
OUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_master.csv'

# Allow notebook to import scripts package
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('PROJECT_ROOT :', PROJECT_ROOT)
print('RAW_DIR      :', RAW_DIR)
print('OUT_PATH     :', OUT_PATH)


PROJECT_ROOT : /Users/arinaali_26/Documents/GitHub/SectionB_G7_BrazilianE-Commerce
RAW_DIR      : /Users/arinaali_26/Documents/GitHub/SectionB_G7_BrazilianE-Commerce/data/raw
OUT_PATH     : /Users/arinaali_26/Documents/GitHub/SectionB_G7_BrazilianE-Commerce/data/processed/cleaned_master.csv


## 2. Run ETL Pipeline

Calls `scripts/etl_pipeline.run_pipeline()` which handles all 7 steps:  
load → parse dates → fix typos → merge → feature engineering → cleanup → save.

In [3]:
from scripts.etl_pipeline import run_pipeline

cleaned_df = run_pipeline(raw_dir=RAW_DIR, output_path=OUT_PATH)

print(f"\nShape returned: {cleaned_df.shape}")

Loading raw data from /Users/arinaali_26/Documents/GitHub/SectionB_G7_BrazilianE-Commerce/data/raw ...
Parsing datetimes ...
Fixing typos ...
Merging tables ...
Engineering features ...
Final cleanup ...
Done → /Users/arinaali_26/Documents/GitHub/SectionB_G7_BrazilianE-Commerce/data/processed/cleaned_master.csv
Rows: 114,092  |  Columns: 43  |  Size: 56.9 MB

Shape returned: (114092, 43)


## 3. Output Validation

Reload the saved file from disk and confirm it matches expectations.  
This catches any silent save/load issues (encoding, index columns, dtype loss).

In [4]:
reloaded = pd.read_csv(OUT_PATH)

print(f"Reloaded shape: {reloaded.shape[0]:,} rows × {reloaded.shape[1]} columns")

# These are the columns downstream notebooks depend on
required_cols = [
    'order_id', 'customer_unique_id', 'customer_state',
    'price', 'freight_value', 'total_item_value',
    'review_score', 'delivery_time_days', 'delay_days', 'is_late',
    'order_year', 'order_month', 'order_month_name', 'order_quarter',
    'product_category_name_english', 'seller_id', 'seller_state',
    'payment_type', 'payment_value', 'payment_installments'
]

missing = [c for c in required_cols if c not in reloaded.columns]
print('Missing required columns:', missing if missing else 'None')
assert len(missing) == 0, f"Missing: {missing}"
print("\n✅ Validation passed.")

Reloaded shape: 114,092 rows × 43 columns
Missing required columns: None

✅ Validation passed.


## 4. Quick Inspection

In [5]:
print("Rows   :", len(reloaded))
print("Columns:", len(reloaded.columns))
reloaded.info()

Rows   : 114092
Columns: 43
<class 'pandas.DataFrame'>
RangeIndex: 114092 entries, 0 to 114091
Data columns (total 43 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       114092 non-null  str    
 1   customer_id                    114092 non-null  str    
 2   order_status                   114092 non-null  str    
 3   order_purchase_timestamp       114092 non-null  str    
 4   order_approved_at              113930 non-null  str    
 5   order_delivered_carrier_date   112112 non-null  str    
 6   order_delivered_customer_date  110839 non-null  str    
 7   order_estimated_delivery_date  114092 non-null  str    
 8   order_item_id                  113314 non-null  float64
 9   product_id                     113314 non-null  str    
 10  seller_id                      113314 non-null  str    
 11  shipping_limit_date            113314 non-null  str    
 12  price        

In [6]:
reloaded.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,order_year,order_month,order_month_name,order_day_of_week,order_quarter,delivery_time_days,estimated_delivery_days,delay_days,is_late,total_item_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,2017,10,Oct,Monday,4,8.0,15,-8.0,0,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,...,2018,7,Jul,Tuesday,3,13.0,19,-6.0,0,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,...,2018,8,Aug,Wednesday,3,9.0,26,-18.0,0,179.12


## 5. Null Summary (Post-Pipeline)

Documenting which nulls remain **after** cleaning and why they are acceptable.

In [7]:
null_counts = reloaded.isnull().sum()
null_pct    = (null_counts / len(reloaded) * 100).round(2)
remaining   = null_counts[null_counts > 0]

if remaining.empty:
    print("No nulls remaining.")
else:
    print(f"  {'Column':<35} {'Nulls':>8}  {'%':>7}")
    print("  " + "-" * 55)
    for col in remaining.index:
        print(f"  {col:<35} {null_counts[col]:>8,}  {null_pct[col]:>6.2f}%")

print("""
Null explanation:
  - order_approved_at / delivered dates : orders that were cancelled before approval
  - product_* columns                   : orders with no items (cancelled/unavailable)
  - review_score                        : orders where customer did not leave a review
  - customer_lat / customer_lng         : zip codes not present in geolocation table
  These are expected and handled in downstream analysis notebooks.
""")

  Column                                 Nulls        %
  -------------------------------------------------------
  order_approved_at                        162    0.14%
  order_delivered_carrier_date           1,980    1.74%
  order_delivered_customer_date          3,253    2.85%
  order_item_id                            778    0.68%
  product_id                               778    0.68%
  seller_id                                778    0.68%
  shipping_limit_date                      778    0.68%
  price                                    778    0.68%
  freight_value                            778    0.68%
  product_weight_g                         778    0.68%
  product_length_cm                        778    0.68%
  product_height_cm                        778    0.68%
  product_width_cm                         778    0.68%
  product_category_name_english            778    0.68%
  seller_zip_code_prefix                   778    0.68%
  seller_city                              778

## Notebook Output

This notebook intentionally writes only one artifact:
data/processed/cleaned_master.csv

**Next notebook in sequence:** `02_cleaning.ipynb` — QA-style cleaning checks and documented data quality decisions.